# Centroid + Feature Mean Tracking

Barebones tracking with windowed cost matrices and Hungarian assignment. Methods: centroid-only, feature-mean-only, centroid plus a small clipped feature-mean correction, and a centroid-feature prototype tracker that keeps a running feature average per active track. No shape terms and no raw-intensity dependency. The search window is not applied to the 3-frame method.

In [5]:
from pathlib import Path
import importlib.util
import json
import sys

import pandas as pd
import torch

repo_root = Path.cwd()
while repo_root != repo_root.parent and not (repo_root / "scripts" / "evaluation" / "feature_mean_tracking.py").exists():
    repo_root = repo_root.parent

module_path = repo_root / "scripts" / "evaluation" / "feature_mean_tracking.py"
if str(module_path.parent) not in sys.path:
    sys.path.insert(0, str(module_path.parent))
spec = importlib.util.spec_from_file_location("feature_mean_tracking", module_path)
feature_mean_tracking = importlib.util.module_from_spec(spec)
sys.modules[spec.name] = feature_mean_tracking
spec.loader.exec_module(feature_mean_tracking)

FeatureMeanConfig = feature_mean_tracking.FeatureMeanConfig
run_feature_mean_tracking = feature_mean_tracking.run_feature_mean_tracking
save_result = feature_mean_tracking.save_result

print(f"repo_root: {repo_root}")
print(f"cuda: {torch.cuda.is_available()} devices={torch.cuda.device_count()}")

repo_root: /nfs/scratch2/inacio/code/llsm/spatialdino/spatialdino
cuda: True devices=1


In [6]:
# Edit these paths before running.
INPUT_PATH = Path("/nfs/scratch2/inacio/data/llsm/spatialdino/model_final/results/bryan/dino/care/642_virus/rope/")
SEGMENTATION_PATH = Path("/nfs/scratch2/inacio/data/llsm/spatialdino/model_final/results/bryan/dino/care/642_virus/seg/")
OUTPUT_PATH = Path("/nfs/scratch2/inacio/data/llsm/spatialdino/model_final/results/bryan/dino/care/642_virus/rope/tracking/")

MAX_FRAMES = None
COMPUTE_GT_METRICS = False

METHODS = ("centroid", "feature_mean", "centroid_feature", "centroid_feature_prototype")
TRACKS_METHOD = "centroid_feature_prototype"
CENTROID_FEATURE_WEIGHT = 0.1
FEATURE_NORM_CLIP = 10.0
Z_WEIGHT = 2.5
MAX_DISTANCE_XY = 20.0
MAX_DISTANCE_Z = 10.0

DEVICE = "cuda:0" if torch.cuda.is_available() else "cpu"

In [7]:
# # Edit these paths before running.
# INPUT_PATH = Path("/nfs/scratch2/inacio/data/llsm/spatialdino/model_final/results/simulated_new/virus_gap/n_600/gap_7/rope/")
# SEGMENTATION_PATH = Path("/nfs/scratch2/inacio/data/llsm/spatialdino/model_final/results/simulated_new/virus_gap/n_600/gap_7/gt/")
# OUTPUT_PATH = Path("/nfs/scratch2/inacio/data/llsm/spatialdino/model_final/results/simulated_new/virus_gap/n_600/gap_7/feature_mean_tracking/")

# MAX_FRAMES = None
# COMPUTE_GT_METRICS = True

# METHODS = ("centroid", "feature_mean", "centroid_feature", "centroid_feature_prototype")
# TRACKS_METHOD = "centroid_feature_prototype"
# CENTROID_FEATURE_WEIGHT = 0.1
# FEATURE_NORM_CLIP = 10.0
# Z_WEIGHT = 2.5
# MAX_DISTANCE_XY = 20.0
# MAX_DISTANCE_Z = 10.0

# DEVICE = "cuda:0" if torch.cuda.is_available() else "cpu"

In [8]:
config = FeatureMeanConfig(
    n_features=384,
    samples_per_object=256,
    seed=12345,
    device=DEVICE,
    methods=METHODS,
    tracks_method=TRACKS_METHOD,
    centroid_feature_weight=CENTROID_FEATURE_WEIGHT,
    feature_norm_clip=FEATURE_NORM_CLIP,
    z_weight=Z_WEIGHT,
    max_distance_xy=MAX_DISTANCE_XY,
    max_distance_z=MAX_DISTANCE_Z,
    max_frames=MAX_FRAMES,
    compute_gt_metrics=COMPUTE_GT_METRICS,
    progress=True,
)

config

FeatureMeanConfig(n_features=384, samples_per_object=256, seed=12345, device='cuda:0', methods=('centroid', 'feature_mean', 'centroid_feature', 'centroid_feature_prototype'), tracks_method='centroid_feature_prototype', centroid_feature_weight=0.1, feature_norm_clip=10.0, z_weight=2.5, max_distance_xy=20.0, max_distance_z=10.0, three_frame_direct_weight=0.25, three_frame_candidate_top_k=8, three_frame_time_limit_seconds=60.0, max_frames=None, compute_gt_metrics=False, progress=True, object_batch_size=512, feature_channel_block=64, sample_batch_size=131072)

In [9]:
result = run_feature_mean_tracking(
    INPUT_PATH,
    SEGMENTATION_PATH,
    config=config,
)

saved_paths = save_result(result, OUTPUT_PATH)
print(json.dumps(saved_paths, indent=2))

[feature-mean-tracking] found 51 frame(s); using 384/390 feature channel(s), 256 sample(s)/object, methods=('centroid', 'feature_mean', 'centroid_feature', 'centroid_feature_prototype'), device=cuda:0


feature means:   0%|                                                                                          …

[feature-mean-tracking] feature means completed in 81.96s


adjacent pairs:   0%|                                                                                         …

prototype pairs:   0%|                                                                                        …

[feature-mean-tracking] matching and tracks completed in 1.13s; total 83.13s
{
  "tracks_csv": "/nfs/scratch2/inacio/data/llsm/spatialdino/model_final/results/bryan/dino/care/642_virus/rope/tracking/tracks.csv",
  "assignments_csv": "/nfs/scratch2/inacio/data/llsm/spatialdino/model_final/results/bryan/dino/care/642_virus/rope/tracking/assignments.csv",
  "timings_csv": "/nfs/scratch2/inacio/data/llsm/spatialdino/model_final/results/bryan/dino/care/642_virus/rope/tracking/timings.csv",
  "config_json": "/nfs/scratch2/inacio/data/llsm/spatialdino/model_final/results/bryan/dino/care/642_virus/rope/tracking/config.json",
  "tracks_centroid_csv": "/nfs/scratch2/inacio/data/llsm/spatialdino/model_final/results/bryan/dino/care/642_virus/rope/tracking/tracks_centroid.csv",
  "tracks_centroid_nested_csv": "/nfs/scratch2/inacio/data/llsm/spatialdino/model_final/results/bryan/dino/care/642_virus/rope/tracking/tracks_by_method/centroid/tracks.csv",
  "tracks_feature_mean_csv": "/nfs/scratch2/inaci

In [113]:
display(result.metrics if result.metrics is not None else pd.DataFrame())
display(result.pair_metrics.head(20) if result.pair_metrics is not None else pd.DataFrame())

,method,frame_pairs,trackable_count,link_pred_count,link_tp,link_fp,link_fn,precision,recall,f1,median_centroid_scale,median_feature_scale,centroid_feature_weight,feature_norm_clip
0,centroid,6,3600,3600,3183,417,417,0.884167,0.884167,0.884167,263.08223,NaN,NaN,NaN
1,feature_mean,6,3600,3600,1844,1756,1756,0.512222,0.512222,0.512222,NaN,0.060884,NaN,NaN
2,centroid_feature,6,3600,3600,3186,414,414,0.885000,0.885000,0.885000,263.08223,0.060884,0.1,10.0
3,centroid_feature_prototype,6,3600,3600,3186,414,414,0.885000,0.885000,0.885000,263.08223,0.055860,0.1,10.0


,method,ref_frame_index,cand_frame_index,ref_timepoint,cand_timepoint,ref_count,cand_count,trackable_count,link_pred_count,link_tp,link_fp,link_fn,precision,recall,f1,centroid_scale,feature_scale,centroid_feature_weight,feature_norm_clip
0,centroid,0,1,00,08,600,600,600,600,495,105,105,0.825000,0.825000,0.825000,262.154449,NaN,NaN,NaN
1,feature_mean,0,1,00,08,600,600,600,600,291,309,309,0.485000,0.485000,0.485000,NaN,0.058122,NaN,NaN
2,centroid_feature,0,1,00,08,600,600,600,600,495,105,105,0.825000,0.825000,0.825000,262.154449,0.058122,0.1,10.0
3,centroid,1,2,08,16,600,600,600,600,541,59,59,0.901667,0.901667,0.901667,262.318420,NaN,NaN,NaN
4,feature_mean,1,2,08,16,600,600,600,600,286,314,314,0.476667,0.476667,0.476667,NaN,0.060270,NaN,NaN
5,centroid_feature,1,2,08,16,600,600,600,600,540,60,60,0.900000,0.900000,0.900000,262.318420,0.060270,0.1,10.0
6,centroid,2,3,16,24,600,600,600,600,530,70,70,0.883333,0.883333,0.883333,262.769379,NaN,NaN,NaN
7,feature_mean,2,3,16,24,600,600,600,600,307,293,293,0.511667,0.511667,0.511667,NaN,0.062683,NaN,NaN
8,centroid_feature,2,3,16,24,600,600,600,600,532,68,68,0.886667,0.886667,0.886667,262.769379,0.062683,0.1,10.0
9,centroid,3,4,24,32,600,600,600,600,539,61,61,0.898333,0.898333,0.898333,263.395081,NaN,NaN,NaN


In [98]:
display(result.assignments.head(50))

,method,ref_frame_index,cand_frame_index,ref_timepoint,cand_timepoint,ref_label,assigned_cand_label,cost,is_true_link,prototype_observations
0,centroid,0,1,00,04,1,1,0.030485,True,NaN
1,centroid,0,1,00,04,2,2,0.035113,True,NaN
2,centroid,0,1,00,04,3,3,0.111295,True,NaN
3,centroid,0,1,00,04,4,4,0.070280,True,NaN
4,centroid,0,1,00,04,5,252,0.130346,False,NaN
5,centroid,0,1,00,04,6,6,0.063838,True,NaN
6,centroid,0,1,00,04,7,7,0.028826,True,NaN
7,centroid,0,1,00,04,8,8,0.113979,True,NaN
8,centroid,0,1,00,04,9,9,0.022566,True,NaN
9,centroid,0,1,00,04,10,102,0.027215,False,NaN


In [99]:
display(result.tracks.head(50))

,track_id,start,t,x,y,z,A,track_length
0,1,1,1,309.000000,223.000000,12.000000,1.0,13
1,1,1,2,316.666656,224.079361,12.460318,1.0,13
2,1,1,3,309.758057,214.370972,9.225806,1.0,13
3,1,1,4,286.432831,225.343277,16.179104,1.0,13
4,1,1,5,289.343750,247.937500,28.500000,1.0,13
5,1,1,6,287.671875,252.109375,36.562500,1.0,13
6,1,1,7,261.671875,236.453125,47.359375,1.0,13
7,1,1,8,246.079361,244.174606,51.269840,1.0,13
8,1,1,9,228.344315,289.781433,63.452095,1.0,13
9,1,1,10,228.590912,311.818176,60.878788,1.0,13
